In [0]:
CATALOG = "arxivist_ai_dev"
GOLD_TABLE = f"{CATALOG}.gold.arxiv_chunks"
VS_ENDPOINT_NAME = "arxiv_vs_endpoint"
VS_INDEX_NAME = f"{CATALOG}.gold.arxiv_chunks_index"

EMBEDDING_MODEL = "databricks-gte-large-en"

print(f"Gold table:       {GOLD_TABLE}")
print(f"VS endpoint:      {VS_ENDPOINT_NAME}")
print(f"VS index:         {VS_INDEX_NAME}")
print(f"Embedding model:  {EMBEDDING_MODEL}")

In [0]:
spark.sql(f"""
    ALTER TABLE {GOLD_TABLE}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

props = spark.sql(f"SHOW TBLPROPERTIES {GOLD_TABLE}").filter("key = 'delta.enableChangeDataFeed'").collect()
print(f"Change Data Feed enabled: {props[0]['value'] if props else 'NOT SET'}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def retrieve(query: str, num_results: int = 5) -> list[dict]:
    """
    Retrieve relevant arXiv paper chunks from Vector Search index.
    
    Args:
        query: Natural language question or search query.
        num_results: Number of top similar chunks to return (default: 5).
    
    Returns:
        List of dicts with keys: chunk_id, title, content, score.
    """
    response = w.vector_search_indexes.query_index(
        index_name=VS_INDEX_NAME,
        columns=["chunk_id", "title", "content", "categories"],
        query_text=query,
        num_results=num_results,
    )

    columns = [col.name for col in response.manifest.columns]
    results = []
    for row in response.result.data_array:
        doc = dict(zip(columns, row))
        results.append({
            "chunk_id": doc.get("chunk_id"),
            "title": doc.get("title"),
            "content": doc.get("content"),
            "categories": doc.get("categories"),
            "score": doc.get("score", 0.0),
        })

    return results


def format_context(results: list[dict]) -> str:
    """
    Format retrieved chunks into a context string for the LLM prompt.
    """
    context_parts = []
    for i, doc in enumerate(results, 1):
        context_parts.append(
            f"[Paper {i}] {doc['title']}\n"
            f"Categories: {doc['categories']}\n"
            f"{doc['content']}"
        )
    return "\n\n---\n\n".join(context_parts)


print("Retrieval functions defined: retrieve(), format_context()")
print(f"  Target index: {VS_INDEX_NAME}")
print(f"  Will return: chunk_id, title, content, categories, score")

In [0]:
from openai import OpenAI

LLM_MODEL = "databricks-meta-llama-3-1-70b-instruct"  
LLM_MAX_TOKENS = 1024
LLM_TEMPERATURE = 0.1  

db_host = spark.conf.get("spark.databricks.workspaceUrl")
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=db_token,
    base_url=f"https://{db_host}/serving-endpoints",
)

SYSTEM_PROMPT = """You are ArxiVist AI — an academic research assistant specializing in computer science.

Your task:
- Answer the user's question using ONLY the provided research paper excerpts below.
- Cite papers by their title when referencing specific findings.
- If the provided context does not contain enough information, say so honestly.
- Be concise but thorough. Use technical language appropriate for a CS audience.

Provided research papers:
{context}
"""

def generate(question: str, context: str) -> str:
    """
    Generate an answer using Foundation Model API (Llama).
    
    Args:
        question: User's natural language question.
        context: Formatted context string from format_context().
    
    Returns:
        LLM-generated answer as a string.
    """
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
            {"role": "user", "content": question},
        ],
        max_tokens=LLM_MAX_TOKENS,
        temperature=LLM_TEMPERATURE,
    )
    return response.choices[0].message.content


print(f"Generation function defined: generate(question, context)")
print(f"  LLM model:     {LLM_MODEL}")
print(f"  Max tokens:    {LLM_MAX_TOKENS}")
print(f"  Temperature:   {LLM_TEMPERATURE}")
print(f"  Cost:          pay-per-token (no endpoint to manage)")

In [0]:
import time

def ask_plain(question: str) -> dict:
    start = time.time()
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful CS research assistant. Answer concisely."},
            {"role": "user", "content": question},
        ],
        max_tokens=LLM_MAX_TOKENS,
        temperature=LLM_TEMPERATURE,
    )
    elapsed = time.time() - start
    return {
        "answer": response.choices[0].message.content,
        "tokens": response.usage.total_tokens,
        "time_sec": round(elapsed, 2),
        "method": "Plain LLM (no context)",
    }


def ask_rag(question: str, num_results: int = 5) -> dict:
    start = time.time()

    docs = retrieve(question, num_results=num_results)
    retrieval_time = time.time() - start

    context = format_context(docs)

    answer = generate(question, context)
    total_time = time.time() - start

    return {
        "answer": answer,
        "sources": [(d["title"], round(d["score"], 4)) for d in docs],
        "num_sources": len(docs),
        "retrieval_time_sec": round(retrieval_time, 2),
        "total_time_sec": round(total_time, 2),
        "method": "RAG (retrieve + generate)",
    }


def compare(question: str, num_results: int = 5):

    print(f"{'='*80}")
    print(f"QUESTION: {question}")
    print(f"{'='*80}")

    # Plain LLM 
    print(f"\n{'─'*40}")
    print(f"METHOD 1: Plain LLM (no retrieval)")
    print(f"{'─'*40}")
    plain = ask_plain(question)
    print(f"\n{plain['answer']}")
    print(f"\n⏱ {plain['time_sec']}s | Tokens: {plain['tokens']}")

    # RAG 
    print(f"\n{'─'*40}")
    print(f"METHOD 2: RAG (Vector Search + LLM)")
    print(f"{'─'*40}")
    rag = ask_rag(question, num_results=num_results)
    print(f"\n{rag['answer']}")
    print(f"\n⏱ {rag['total_time_sec']}s (retrieval: {rag['retrieval_time_sec']}s)")
    print(f"Sources ({rag['num_sources']} papers):")
    for title, score in rag["sources"]:
        print(f"  • [{score}] {title}")

    print(f"\n{'='*80}")
    return {"plain": plain, "rag": rag}


print("Inference functions defined:")
print("  ask_plain(question)     → LLM only (baseline)")
print("  ask_rag(question)       → retrieve + generate (RAG)")
print("  compare(question)       → side-by-side comparison")

In [0]:
from databricks.sdk.service.vectorsearch import EndpointType

existing = [ep for ep in w.vector_search_endpoints.list_endpoints() if ep.name == VS_ENDPOINT_NAME]

if existing:
    ep = existing[0]
    print(f"Endpoint '{VS_ENDPOINT_NAME}' already exists (status: {ep.endpoint_status.state.value})")
else:
    print(f"Creating endpoint '{VS_ENDPOINT_NAME}'... (this takes ~5-10 min to provision)")
    ep = w.vector_search_endpoints.create_endpoint_and_wait(
        name=VS_ENDPOINT_NAME,
        endpoint_type=EndpointType.STANDARD
    )
    print(f"Endpoint provisioned!")

ep_info = w.vector_search_endpoints.get_endpoint(VS_ENDPOINT_NAME)
print(f"\nEndpoint details:")
print(f"  Name:   {ep_info.name}")
print(f"  Status: {ep_info.endpoint_status.state.value}")

In [0]:
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
    VectorIndexType,
)

try:
    idx = w.vector_search_indexes.get_index(VS_INDEX_NAME)
    print(f"Index '{VS_INDEX_NAME}' already exists (status: {idx.status})")
except Exception:
    print(f"Creating Delta Sync index '{VS_INDEX_NAME}'...")
    print(f"  Source table:     {GOLD_TABLE}")
    print(f"  Primary key:      chunk_id")
    print(f"  Embedding column: content")
    print(f"  Embedding model:  {EMBEDDING_MODEL}")
    print(f"  Sync mode:        TRIGGERED")
    print()

    idx = w.vector_search_indexes.create_index(
        index_type=VectorIndexType.DELTA_SYNC,
        name=VS_INDEX_NAME,
        endpoint_name=VS_ENDPOINT_NAME,
        primary_key="chunk_id",
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=GOLD_TABLE,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="content",
                    embedding_model_endpoint_name=EMBEDDING_MODEL,
                )
            ],
        ),
    )
    print(f"Index creation initiated!")

import time as _time

print(f"\nWaiting for index to become ONLINE...")
for attempt in range(60):
    idx = w.vector_search_indexes.get_index(VS_INDEX_NAME)
    status = idx.status
    if status.ready:
        print(f"\n\u2705 Index is ONLINE and ready! (took ~{attempt * 30}s)")
        break
    print(f"  [{attempt+1}] Status: {status.message or 'provisioning...'}", end="\r")
    _time.sleep(30)
else:
    print(f"\n\u26a0\ufe0f  Timeout: index still not ready after 30 min. Check manually.")